# Notebook 02 — Preprocesamiento y Split Train/Val/Test
**Proyecto:** Sistema Multi-Agente para Análisis de Deserción Estudiantil

---
## Objetivos
1. Cargar los datos crudos del notebook 01
2. Limpiar y preparar features
3. Dividir en train/val/test **sin data leakage** (el split ocurre ANTES que cualquier transformación)
4. Guardar los splits en `data/processed/` para que el notebook 03 los use directamente

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
import warnings
import os

warnings.filterwarnings('ignore')

RAW_PATH = '../data/raw/student_dropout.csv'
PROCESSED_PATH = '../data/processed/'
os.makedirs(PROCESSED_PATH, exist_ok=True)

print('Librerías cargadas.')

Librerías cargadas.


## 1. Carga y binarización de la variable objetivo

In [2]:
df = pd.read_csv(RAW_PATH)

# Binarizar target: Dropout=1, resto=0
df['Dropout'] = (df['Target'] == 'Dropout').astype(int)
df = df.drop(columns=['Target'])

print(f'Dataset cargado: {df.shape}')
print(f'Tasa de deserción: {df["Dropout"].mean()*100:.1f}%')

Dataset cargado: (4424, 37)
Tasa de deserción: 32.1%


## 2. Separar features y target

In [3]:
X = df.drop(columns=['Dropout'])
y = df['Dropout']

print(f'X: {X.shape} | y: {y.shape}')
print(f'Clases — 0 (No Dropout): {(y==0).sum()} | 1 (Dropout): {(y==1).sum()}')

X: (4424, 36) | y: (4424,)
Clases — 0 (No Dropout): 3003 | 1 (Dropout): 1421


## 3. Split Train / Val / Test

> **IMPORTANTE — Sin data leakage:**  
> El split ocurre aquí, ANTES de cualquier normalización o imputación.  
> La normalización se aplica dentro del pipeline en el notebook 03, ajustando el scaler **solo sobre train**.

Proporción: **70% train | 15% val | 15% test**  
Se usa `stratify=y` para mantener la proporción de clases en cada split.

In [4]:
# Primer split: 70% train, 30% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

# Segundo split: 50/50 del temp → 15% val, 15% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train: {len(X_train)} muestras ({len(X_train)/len(X)*100:.0f}%)')
print(f'Val:   {len(X_val)} muestras ({len(X_val)/len(X)*100:.0f}%)')
print(f'Test:  {len(X_test)} muestras ({len(X_test)/len(X)*100:.0f}%)')
print(f'\nTasa deserción — Train: {y_train.mean()*100:.1f}% | Val: {y_val.mean()*100:.1f}% | Test: {y_test.mean()*100:.1f}%')

Train: 3096 muestras (70%)
Val:   664 muestras (15%)
Test:  664 muestras (15%)

Tasa deserción — Train: 32.1% | Val: 32.2% | Test: 32.1%


## 4. Guardar splits en disco

In [5]:
def guardar_split(X, y, nombre, ruta):
    split = X.copy()
    split['Dropout'] = y.values
    ruta_archivo = os.path.join(ruta, f'{nombre}.csv')
    split.to_csv(ruta_archivo, index=False)
    print(f'Guardado: {ruta_archivo}')


guardar_split(X_train, y_train, 'train', PROCESSED_PATH)
guardar_split(X_val,   y_val,   'val',   PROCESSED_PATH)
guardar_split(X_test,  y_test,  'test',  PROCESSED_PATH)

print('\nSplits guardados correctamente.')
print('Próximo paso: notebook 03_modeling.ipynb')

Guardado: ../data/processed/train.csv
Guardado: ../data/processed/val.csv
Guardado: ../data/processed/test.csv

Splits guardados correctamente.
Próximo paso: notebook 03_modeling.ipynb
